<a href="https://colab.research.google.com/github/dgaida/wpf_dlml_th_public/blob/main/assets/exercises/week_09/04_LIME_nlp_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LIME für NLP mit Embeddings und Deep Learning

In diesem Notebook aktualisieren wir das vorherige Beispiel und nutzen anstelle von TF-IDF und Random Forest ein Embedding-Modell von Hugging Face gefolgt von einem neuronalen Netzwerk zur binären Klassifikation von Texten (Atheismus vs. Christentum).

In [ ]:
!pip install scikit-learn lime sentence-transformers tensorflow

In [ ]:
import lime
import sklearn
import sklearn.metrics
import numpy as np
import tensorflow as tf
from sentence_transformers import SentenceTransformer
from lime.lime_text import LimeTextExplainer
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping

## Daten laden und vorbereiten

Wir nutzen den 20 Newsgroups Datensatz und beschränken uns auf zwei Kategorien: Atheismus und Christentum.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
categories = ['alt.atheism', 'soc.religion.christian']
newsgroups_train = fetch_20newsgroups(subset='train', categories=categories)
newsgroups_test = fetch_20newsgroups(subset='test', categories=categories)
class_names = ['atheism', 'christian']

## Hugging Face Embedding Modell

Wir nutzen ein vortrainiertes Modell von Hugging Face (`all-MiniLM-L6-v2`), um die Texte in numerische Vektoren umzuwandeln.

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print("Erzeuge Embeddings für Trainingsdaten...")
train_embeddings = embed_model.encode(newsgroups_train.data, show_progress_bar=True)
print("Erzeuge Embeddings für Testdaten...")
test_embeddings = embed_model.encode(newsgroups_test.data, show_progress_bar=True)

## Training eines neuronalen Netzwerks

Wir erstellen ein einfaches Feed-Forward-Netzwerk mit 2-3 Dense-Layern.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(train_embeddings.shape[1],)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    #tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Define EarlyStopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(train_embeddings, newsgroups_train.target,
                    epochs=25,
                    batch_size=32,
                    validation_split=0.1,
                    verbose=1,
                    callbacks=[early_stopping])

In [ ]:
loss, accuracy = model.evaluate(test_embeddings, newsgroups_test.target)
print(f"\nTest Accuracy: {accuracy:.4f}")

## Erklärungen mit LIME

LIME (Local Interpretable Model-agnostic Explanations) hilft uns zu verstehen, welche Wörter im Text die Entscheidung des Modells am stärksten beeinflusst haben.

In [ ]:
def predict_proba(texts):
    # LIME übergibt eine Liste von Texten (Strings)
    embeddings = embed_model.encode(texts, show_progress_bar=False)
    return model.predict(embeddings, verbose=0)

In [ ]:
explainer = LimeTextExplainer(class_names=class_names)

idx = 84 # Beispiel-Index
exp = explainer.explain_instance(newsgroups_test.data[idx], predict_proba, num_features=6)

print('Dokument ID: %d' % idx)
print('Wahrscheinlichkeit(christian) =', predict_proba([newsgroups_test.data[idx]])[0, 1])
print('Wahre Klasse: %s' % class_names[newsgroups_test.target[idx]])

## Visualisierung

In [ ]:
%matplotlib inline
fig = exp.as_pyplot_figure()
plt.title(f"LIME Erklärung für Dokument {idx}")
plt.show()

In [ ]:
exp.show_in_notebook(text=True)